# **Two Tower using Metadata**

# Data Loading


In [ ]:
import pandas as pd

anime_filtered = pd.read_csv('/content/anime_10pct.csv')
users_filtered = pd.read_csv('/content/users_10pct.csv')
ratings_filtered= pd.read_csv('/content/ratings_10pct.csv')


# Initial Analysis
print(f"anime_filtered.shape: {anime_filtered.shape}")
print(f"users_filtered.shape: {users_filtered.shape}")
print(f"ratings_filtered.shape: {ratings_filtered.shape}")

# Check for columns
print(f"anime_filtered.columns: {anime_filtered.columns}")
print(f"users_filtered.columns: {users_filtered.columns}")
print(f"ratings_filtered.columns: {ratings_filtered.columns}")

# Rename columns for consistency
users_filtered = users_filtered.rename(columns={'Mal_ID': 'user_id'})

# Fill missing sypnosis in anime_filtered
anime_filtered['sypnopsis'] = anime_filtered['sypnopsis'].fillna("No description available")

anime_filtered.shape: (14681, 25)
users_filtered.shape: (74951, 17)
ratings_filtered.shape: (2029590, 4)
anime_filtered.columns: Index(['anime_id', 'Name', 'Score', 'Genres', 'English_name', 'Japanese_name',
       'sypnopsis', 'Type', 'Episodes', 'Aired', 'Premiered', 'Producers',
       'Licensors', 'Studios', 'Source', 'Duration', 'Rating', 'Ranked',
       'Popularity', 'Members', 'Favorites', 'Watching', 'Completed',
       'On-Hold', 'Dropped'],
      dtype='object')
users_filtered.columns: Index(['Mal_ID', 'Username', 'Gender', 'Birthday', 'Location', 'Joined',
       'Days_Watched', 'Mean_Score', 'Watching', 'Completed', 'On_Hold',
       'Dropped', 'Plan_to_Watch', 'Total_Entries', 'Rewatched',
       'Episodes_Watched', 'mean_anime_popularity'],
      dtype='object')
ratings_filtered.columns: Index(['user_id', 'anime_id', 'rating', 'Popularity'], dtype='object')


# Installations

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

!pip install tensorflow tensorflow_recommenders
import tensorflow as tf
import tensorflow_recommenders as tfrs

import tf_keras
print(f"Legacy Keras version: {tf_keras.__version__}")

import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score, log_loss

Legacy Keras version: 2.19.0


# Model Training

## Section 1: Prepare & Clean Data
The goal is to transform the raw dataset into a structured format.

1. Filtering by Ratings (>=7)

In Recommendation Systems, a "watch" does not always suggest a "like". By filtering for ratings >= 7, we are ensuring that the model  learns what users actually enjoyed, rather than just what they consumed.

2. Missing Value Imputation

We will be using empty strings `""` for text and `0` for numerics. This is because neural networks cannot handle `NaN` values as this would cause the gradient calculation to collapse to `NaN`.

In [ ]:
# =====================================================
# 1. PREPARE & CLEAN DATA
# =====================================================
ratings = ratings_filtered.copy()
ratings = ratings[ratings["rating"] >= 7]

# Merge features
ratings = ratings.merge(users_filtered[["user_id", "Gender", "mean_anime_popularity", "Mean_Score"]], on="user_id", how="left")
ratings = ratings.merge(anime_filtered[["anime_id", "sypnopsis", "Genres", "Studios", "Source"]], on="anime_id", how="left")

ratings.columns = [c.lower() for c in ratings.columns]

string_cols = ["user_id", "anime_id", "gender", "sypnopsis", "genres", "studios", "source"]
for col in string_cols:
    ratings[col] = ratings[col].fillna("").astype(str)

ratings["genres"] = ratings["genres"].str.replace(",", " ")
ratings["mean_anime_popularity"] = ratings["mean_anime_popularity"].fillna(0).astype(np.float32)
ratings["mean_score"] = ratings["mean_score"].fillna(0).astype(np.float32)

# Global Candidates
unique_anime_df = anime_filtered[["anime_id", "sypnopsis", "Genres", "Studios", "Source"]].copy()
unique_anime_df.columns = [c.lower() for c in unique_anime_df.columns]
for col in unique_anime_df.columns:
    unique_anime_df[col] = unique_anime_df[col].fillna("").astype(str)
unique_anime_df["genres"] = unique_anime_df["genres"].str.replace(",", " ")
unique_anime_df = unique_anime_df.drop_duplicates()

candidate_ds = tf.data.Dataset.from_tensor_slices(dict(unique_anime_df))

## Section 2: Feature Normalization & Vectorization

The goal is to use domain (Anime-specific) knowledge to help the model distinguish between the different types of content.

1. Numeric Normalization

`mean_anime_popularity` represents the average ranking of an anime relative to all other titles, with values ranging approximately from 1 to 14,000. In contrast, `mean_score` denotes the average rating assigned by a user across their reviews, bounded between 0 and 10. Due to this substantial difference in scale, `mean_anime_popularity` would dominate the feature space if used directly, disproportionately influencing distance-based computations and model behavior.

To address this issue, both features are standardized to have a mean of 0 and a variance of 1. This normalization ensures that each feature contributes comparably to the final vector representation, preventing larger-scale variables from overwhelming smaller-scale ones and improving numerical stability and model performance.

2. Multi-Hot Genre Encoding

Simple label encoding (e.g., 1, 2, 3) introduces an artificial ordering between genres that does not actually exist. For instance, encoding "Action" as 1 and "Sci-Fi" as 2 would incorrectly imply that one genre is greater than or closer to another. Instead, multi-hot encoding represents each genre as an independent binary feature. This allows multiple genres to be active at the same time, enabling the model to capture that an anime can simultaneously belong to categories such as "Action" and "Sci-Fi" without imposing any unintended relationships.

3. Vectorization of Synopsis

The synopsis text often contains important contextual details about themes, tone, and story elements. To capture this information effectively, a vocabulary size of 15,000 tokens is used. This is large enough to include meaningful but less frequent words, while filtering out extremely rare terms that are likely to introduce noise. In addition, the sequence length is capped at 200 words, since the most important plot details are typically introduced early in the synopsis. This helps retain the most informative content while keeping the input size manageable.

In [ ]:
# =====================================================
# 2. FEATURE NORMALIZATION & VECTORIZATION
# =====================================================
pop_norm = tf.keras.layers.Normalization(axis=None)
pop_norm.adapt(ratings["mean_anime_popularity"].values)

score_norm = tf.keras.layers.Normalization(axis=None)
score_norm.adapt(ratings["mean_score"].values)

genre_vectorizer = tf.keras.layers.TextVectorization(max_tokens=200, output_mode='multi_hot')
genre_vectorizer.adapt(candidate_ds.map(lambda x: x["genres"]).batch(1024))

synopsis_vectorizer = tf.keras.layers.TextVectorization(max_tokens=15000, output_sequence_length=200)
synopsis_vectorizer.adapt(candidate_ds.map(lambda x: x["sypnopsis"]).batch(1024))

## Section 3: Towers

### User Tower

**Input Features**
- **`user_id`**: Acts as the primary anchor for learning individual user preferences. It allows the model to capture personalized viewing patterns that may not be reflected in metadata alone.  
- **`gender`**: Anime preferences often show statistical correlations with demographic groups (e.g., Shonen vs. Shojo audiences). Including gender provides an additional signal for preference modeling.  
- **`mean_anime_popularity`**: Indicates whether a user tends to prefer mainstream titles (high popularity) or niche, lesser-known shows. This helps the model differentiate between users who follow trends and those who explore less popular content.  
- **`mean_score`**: Serves as a bias-correction feature. It helps distinguish between stricter users who rate everything lower and more generous users who consistently give higher scores.  

**Architecture Choices**

- **User ID Embedding (128 dimensions)**  
  **Justification:** A 128-dimensional embedding provides a good balance for datasets with millions of interactions. It is large enough to capture complex preference patterns while remaining computationally efficient and avoiding excessive memory usage.

- **Dropout (0.3)**  
  **Justification:** Dropout randomly disables 30% of neurons during training. This prevents the model from relying too heavily on the `user_id` embedding alone and encourages it to also learn from metadata features such as gender and mean score. This improves generalization.

---

### Anime Tower

**Input Features**
- **`anime_id`**: Allows the model to learn the unique characteristics of each show that may not be fully captured by structured metadata.  
- **`synopsis`**: Provides rich textual information and is particularly useful for cold-start scenarios. The model can recommend new or unseen anime based on plot themes and keywords.  
- **`genres`**: One of the strongest categorical predictors. Users who enjoy a specific genre (e.g., Psychological) are more likely to prefer other anime with similar tags.  
- **`studios`**: Many viewers follow specific studios due to consistent art styles or production quality (e.g., action-focused vs. slice-of-life storytelling).  
- **`source`**: The original source material (e.g., manga, light novel, original) often influences pacing, tone, and storytelling style, making it a useful predictive feature.

**Architecture Choices**

- **Bidirectional LSTM (32 units)**  
  **Logic:** Standard LSTMs process text from left to right, while Bidirectional LSTMs read the sequence in both directions.  
  **Justification:** Important contextual cues in an anime synopsis may appear at the beginning or the end. Reading in both directions allows the model to better understand relationships between words and capture overall tone more effectively.

- **L2 Regularization (1e-6)**  
  **Justification:** L2 regularization penalizes large weights during training. This prevents embeddings from becoming overly extreme and encourages a smoother vector space, ensuring that similar anime remain close together in the learned representation.

In [ ]:
# =====================================================
# 3. TOWERS
# =====================================================
class UserModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.user_id_lookup = tf.keras.layers.StringLookup(vocabulary=ratings["user_id"].unique(), mask_token=None)

        # Convert user_id into a 128-dimensional embedding vector with L2 regularization
        self.user_emb = tf.keras.layers.Embedding(
            len(ratings["user_id"].unique()) + 1, 128,
            embeddings_regularizer=tf.keras.regularizers.l2(1e-6)
        )

        # Convert gender into a 16-dimensional embedding vector
        self.gender_lookup = tf.keras.layers.StringLookup(vocabulary=ratings["gender"].unique(), mask_token=None)
        self.gender_emb = tf.keras.layers.Embedding(len(ratings["gender"].unique()) + 1, 16)

        # Convert the normalized mean_anime_popularity and mean_score into a 32-dimensional dense representation
        self.numeric_dense = tf.keras.layers.Dense(32, activation="relu")

        # Compress the combined features into a 64-dimensional embedding vector with a hidden layer and dropout for regularization
        self.projection = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(0.3), # Prevents overfitting
            tf.keras.layers.Dense(64)
        ])

    def call(self, inputs):
        p = pop_norm(tf.cast(inputs["mean_anime_popularity"], tf.float32))
        s = score_norm(tf.cast(inputs["mean_score"], tf.float32))
        numeric_feats = tf.stack([p, s], axis=1)

        user_id_emb = self.user_emb(self.user_id_lookup(inputs["user_id"]))
        gender_emb = self.gender_emb(self.gender_lookup(inputs["gender"]))

        return self.projection(tf.concat([
            user_id_emb,
            gender_emb,
            self.numeric_dense(numeric_feats)
        ], axis=1))

class AnimeModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.id_lookup = tf.keras.layers.StringLookup(vocabulary=unique_anime_df["anime_id"].unique(), mask_token=None)

        # Convert anime_id into a 128-dimensional embedding vector with L2 regularization
        self.id_emb = tf.keras.layers.Embedding(
            len(unique_anime_df["anime_id"].unique()) + 1, 128,
            embeddings_regularizer=tf.keras.regularizers.l2(1e-6)
        )

        # Process synopsis with an embedding layer followed by a bidirectional LSTM of 64 dimensions (32 in each direction)
        self.syn_emb = tf.keras.Sequential([
            synopsis_vectorizer,
            tf.keras.layers.Embedding(15001, 128, mask_zero=True),
            tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32))
        ])

        # Process genres with a multi-hot vector followed by a dense layer to reduce dimensionality to 32
        self.gen_emb = tf.keras.Sequential([
            genre_vectorizer,
            tf.keras.layers.Dense(32, activation="relu")
        ])

        # Compress the combined features into a 64-dimensional embedding vector with a hidden layer and dropout for regularization
        self.projection = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(64)
        ])

    def call(self, inputs):
        id_emb = self.id_emb(self.id_lookup(inputs["anime_id"]))
        return self.projection(tf.concat([
            id_emb,
            self.syn_emb(inputs["sypnopsis"]),
            self.gen_emb(inputs["genres"])
        ], axis=1))

class TwoTowerModel(tfrs.Model):
    def __init__(self, user_model, anime_model):
        super().__init__()
        self.user_model = user_model
        self.anime_model = anime_model
        self.task = tfrs.tasks.Retrieval()

    def compute_loss(self, features, training=False):
        user_embeddings = self.user_model(features)
        anime_embeddings = self.anime_model(features)
        return self.task(user_embeddings, anime_embeddings, compute_metrics=not training)

## Section 4: Training with Scheduler

### Training Parameters

**Optimizer: Adam**  
**Justification:** Adam is an adaptive optimization algorithm that adjusts the learning rate for each parameter individually. This is especially useful in recommender systems, where different features learn at different speeds. For example, common genres may receive frequent updates, while rare user IDs or niche anime receive fewer updates. Adam helps balance these differences and typically leads to more stable convergence.

**Learning Rate: 0.0003**  
**Justification:** A common default learning rate is 0.001, but a smaller value (0.0003) is used here because LSTM-based text encoders are more sensitive to large updates. A high learning rate can cause the model to overshoot optimal values, leading to unstable training and fluctuating loss. Using a lower learning rate results in smoother and more stable convergence.

**Batch Size: 2048**  
**Justification:** In retrieval-based training, other samples within the same batch act as negative examples. With a batch size of 2048, each positive interaction is contrasted against 2,047 negative samples. This large number of negatives strengthens the learning signal, improves representation quality, and makes training more efficient.

**Loss Function: Retrieval Task (Cross-Entropy)**  
**Justification:** The problem is formulated as a multiclass classification task, where the correct class corresponds to the anime that the user actually interacted with. Cross-entropy loss encourages the model to assign higher similarity scores to true user–anime pairs while pushing unrelated pairs further apart in the embedding space.

**Epochs: 20**  
**Justification:** The model is trained for 20 epochs to allow sufficient exposure to the full dataset while avoiding overfitting. Retrieval models with large embedding spaces typically require multiple passes through the data to learn meaningful user–item relationships, especially when both towers contain several features and text encoders. Empirically, around 15–25 epochs often provides a good balance, where the model has enough time to converge but has not yet started memorizing noise. Training beyond this point usually results in diminishing returns and may increase the risk of overfitting, while fewer epochs may lead to undertrained embeddings.

In [ ]:
# =====================================================
# 4. TRAINING WITH SCHEDULER
# =====================================================
model = TwoTowerModel(UserModel(), AnimeModel())

model.compile(optimizer=tf.keras.optimizers.Adam(0.0003))

# Calculate split sizes
num_samples = len(ratings)
train_size = int(0.8 * num_samples)
val_size = int(0.1 * num_samples)
test_size = num_samples - train_size - val_size

# Create the sets
full_ds = tf.data.Dataset.from_tensor_slices(dict(ratings)).shuffle(100_000)

train_ds = full_ds.take(train_size).batch(2048).cache()
val_ds = full_ds.skip(train_size).take(val_size).batch(2048).cache()
test_ds = full_ds.skip(train_size + val_size).take(test_size).batch(2048).cache()

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)

Epoch 1/20
337/337 [==============================] - 530s 2s/step - loss: 15446.3583 - regularization_loss: 0.0086 - total_loss: 15446.3668 - val_loss: 1193.7731 - val_regularization_loss: 0.0081 - val_total_loss: 1193.7812
Epoch 2/20
337/337 [==============================] - 498s 1s/step - loss: 15161.0993 - regularization_loss: 0.0084 - total_loss: 15161.1077 - val_loss: 1157.6039 - val_regularization_loss: 0.0088 - val_total_loss: 1157.6127
Epoch 3/20
337/337 [==============================] - 505s 1s/step - loss: 14709.3735 - regularization_loss: 0.0092 - total_loss: 14709.3827 - val_loss: 1128.7859 - val_regularization_loss: 0.0097 - val_total_loss: 1128.7955
Epoch 4/20
337/337 [==============================] - 487s 1s/step - loss: 14331.4702 - regularization_loss: 0.0101 - total_loss: 14331.4803 - val_loss: 1120.9569 - val_regularization_loss: 0.0106 - val_total_loss: 1120.9675
Epoch 5/20
337/337 [==============================] - 507s 2s/step - loss: 14051.2874 - regularizati

## Section 5: Global Evaluation

The model is evaluated using ranking-based retrieval metrics at **K = 10**. For each user in the test set, the model computes similarity scores between the user embedding and candidate anime embeddings. The top-10 highest scoring anime are selected, and the metrics measure whether the **ground truth anime (the one actually watched)** appears in this list and how highly it is ranked.

Since each test example has **only one correct anime**, these metrics effectively evaluate how well the model retrieves and ranks that single relevant item.

---

### Precision@10
Precision@10 measures whether the correct anime appears in the top 10, normalized by the list size.

Because there is only **one relevant item**, Precision@10 can only be:
- **0.1** → correct anime is in top 10  
- **0.0** → correct anime is not in top 10  

**Interpretation:**  
- If **Precision@10 = 0.082**, this means **8.2% of recommendations in the top 10 are correct on average**, which corresponds to the model finding the correct anime about **8.2% of the time**.

---

### Recall@10
Recall@10 measures whether the model successfully retrieves the correct anime.

Since there is **only one relevant item**, Recall@10 becomes:

- **1.0** → correct anime found in top 10  
- **0.0** → not found  

**Interpretation:**  
- If **Recall@10 = 0.42**, it means the model retrieves the correct anime **42% of the time**.  
- In this setup, **Recall@10 is equivalent to HitRate@10**.

---

### HitRate@10
HitRate@10 checks whether the correct anime appears anywhere in the top 10 list.

**Interpretation:**  
- If **HitRate@10 = 0.65**, this means **65% of users see the correct anime within the first 10 recommendations**.  
- This is often the most important **business metric**, since users typically only look at the first few suggestions.

---

### MAP@10 (Mean Average Precision)
MAP@10 evaluates **both correctness and ranking position**. If the correct anime appears earlier in the list, the score is higher.

For a single relevant item:

- Rank 1 → AP = 1.0  
- Rank 2 → AP = 0.5  
- Rank 5 → AP = 0.2  
- Rank 10 → AP = 0.1  
- Not in top 10 → AP = 0  

**Interpretation:**  
- If **MAP@10 = 0.28**, it means the correct anime is usually found **around rank 3–4 on average**.  
- Higher MAP means the model places correct recommendations closer to the top.

---

### NDCG@10 (Normalized Discounted Cumulative Gain)
NDCG@10 measures ranking quality by giving **more weight to higher positions**.

This means:
- Rank 1 → 1.000  
- Rank 2 → 0.630  
- Rank 3 → 0.500  
- Rank 5 → 0.387  
- Rank 10 → 0.289  

**Interpretation:**  
- If **NDCG@10 = 0.52**, it means correct anime are typically ranked **within the top few positions**, not near the bottom.  
- If HitRate is high but NDCG is low, the model finds correct anime but ranks them too low.

In [ ]:
# =====================================================
# 5. GLOBAL EVALUATION
# =====================================================
print("\nBuilding candidate embeddings for full catalog...")
candidate_embeddings = []
candidate_ids_list = []

for batch in candidate_ds.batch(1024):
    emb = model.anime_model(batch)
    candidate_embeddings.append(emb)
    # Decode byte-strings to standard strings for matching
    ids = [i.decode('utf-8') for i in batch["anime_id"].numpy()]
    candidate_ids_list.extend(ids)

candidate_embeddings = tf.concat(candidate_embeddings, axis=0)
candidate_ids_array = np.array(candidate_ids_list)

def evaluate_ranking_metrics(model, test_dataset, k):

    precisions = []
    recalls = []
    hit_rates = []
    aps = []
    ndcgs = []
    bce_losses = []
    auc_scores = []

    batched_test = test_dataset.batch(512)

    for batch in batched_test:

        user_emb = model.user_model(batch)
        item_emb = model.anime_model(batch)

        scores = tf.matmul(user_emb, item_emb, transpose_b=True)
        scores = scores.numpy()

        batch_size = scores.shape[0]

        probs = tf.nn.sigmoid(scores).numpy()
        y_true = np.eye(batch_size)

        # BCE Loss
        batch_bce = log_loss(y_true.ravel(), probs.ravel(), labels=[0, 1])
        bce_losses.append(batch_bce)

        # AUC Score
        batch_auc = roc_auc_score(y_true.ravel(), probs.ravel())
        auc_scores.append(batch_auc)

        for i in range(batch_size):

            ranking = np.argsort(-scores[i])[:k]

            # Ground truth is diagonal (i-th item is correct)
            relevant = [i]

            # Precision@K
            precision = len(set(ranking) & set(relevant)) / k
            precisions.append(precision)

            # Recall@K
            recall = len(set(ranking) & set(relevant)) / len(relevant)
            recalls.append(recall)

            # HitRate@K
            hit = 1 if i in ranking else 0
            hit_rates.append(hit)

            # Average Precision@K
            ap = 0
            hits = 0
            for j, item in enumerate(ranking):
                if item == i:
                    hits += 1
                    ap += hits / (j + 1)

            ap = ap / 1
            aps.append(ap)

            # NDCG@K
            dcg = 0
            for j, item in enumerate(ranking):
                if item == i:
                    dcg += 1 / np.log2(j + 2)

            idcg = 1.0
            ndcgs.append(dcg / idcg)

    print(f"\nEvaluation @ {k}")
    print("--------------------------")
    print(f"BCE Loss:    {np.mean(bce_losses):.4f}")
    print(f"AUC Score:   {np.mean(auc_scores):.4f}")
    print(f"Precision@{k}: {np.mean(precisions):.4f}")
    print(f"Recall@{k}: {np.mean(recalls):.4f}")
    print(f"HitRate@{k}: {np.mean(hit_rates):.4f}")
    print(f"MAP@{k}: {np.mean(aps):.4f}")
    print(f"NDCG@{k}: {np.mean(ndcgs):.4f}")


Building candidate embeddings for full catalog...


In [ ]:
# Evaluation at K = 5
# Unbatch the test dataset for evaluation
evaluate_ranking_metrics(model, test_ds.unbatch(), k=5)


Evaluation @ 5
--------------------------
BCE Loss:    5.3395
AUC Score:   0.6260
Precision@5: 0.0238
Recall@5: 0.1191
HitRate@5: 0.1191
MAP@5: 0.0739
NDCG@5: 0.0850


In [ ]:
# Evaluation at K = 10
# Unbatch the test dataset for evaluation
evaluate_ranking_metrics(model, test_ds.unbatch(), k=10)


Evaluation @ 10
--------------------------
BCE Loss:    5.3395
AUC Score:   0.6260
Precision@10: 0.0170
Recall@10: 0.1695
HitRate@10: 0.1695
MAP@10: 0.0805
NDCG@10: 0.1012
